In [1]:
#setup

from quantum_lake_student.config import Settings
from quantum_lake_student.connections import minio_client, bronze_inventory

settings = Settings.from_environment()
inventory = dict(bronze_inventory(settings))   # {object_name: size_bytes}
for name, size in sorted(inventory.items()):
    print(f"{size:>12,}  {name}")


  14,638,673  bronze/source=google_qec/google-surface-code-curated.zip
     144,172  bronze/source=qasmbench/qasmbench-qec.zip
     358,017  bronze/source=qec_syndromes/syndromes_dataset.zip


In [2]:
#the expected truth /manifest

import json
from pathlib import Path

manifest = json.loads(Path("/course-data/metadata/bundle-manifest.json").read_text())
for obj in manifest["objects"]:
    print(obj["source"], obj["path"], obj["bytes"], obj["sha256"][:12] + "…")


qasmbench raw/source=qasmbench/qasmbench-qec.zip 144172 60307f88e34b…
qec_syndromes raw/source=qec_syndromes/syndromes_dataset.zip 358017 bdfce36a71f0…
google_qec raw/source=google_qec/google-surface-code-curated.zip 14638673 5d6a24f89f05…


In [3]:
#we verify the checksums to see if the match the expected size

import hashlib

def sha256_of_object(object_name):
    client = minio_client(settings)
    resp = client.get_object(settings.s3_bucket, object_name)
    try:
        data = resp.read()
    finally:
        resp.close(); resp.release_conn()
    return hashlib.sha256(data).hexdigest(), data

manifest_by_source = {o["source"]: o for o in manifest["objects"]}
raw_bytes = {}   # keep for cell 4/5

for object_name in inventory:
    source = object_name.split("source=")[1].split("/")[0]
    expected = manifest_by_source[source]
    digest, data = sha256_of_object(object_name)
    raw_bytes[object_name] = data
    ok_hash = digest == expected["sha256"]
    ok_size = len(data) == expected["bytes"]
    print(source, "hash OK" if ok_hash else "HASH MISMATCH!",
                  "size OK" if ok_size else "SIZE MISMATCH!")


google_qec hash OK size OK
qasmbench hash OK size OK
qec_syndromes hash OK size OK


In [4]:
expected_names = {
    o["path"].replace("raw/", "bronze/", 1) for o in manifest["objects"] if o["mandatory"]
}
actual_names = set(inventory)

missing = expected_names - actual_names
unexpected = actual_names - expected_names

print("missing:", missing or "none")
print("unexpected:", unexpected or "none")


missing: none
unexpected: none


In [5]:
#we list the sources and the members

import io, zipfile

def is_unsafe_member(name: str) -> bool:
    n = name.replace("\\", "/")
    parts = n.split("/")
    return n.startswith("/") or ".." in parts or ":" in parts[0]

for object_name, data in raw_bytes.items():
    with zipfile.ZipFile(io.BytesIO(data)) as zf:
        members = zf.namelist()
    unsafe = [m for m in members if is_unsafe_member(m)]
    print(f"\n{object_name}: {len(members)} members, {len(unsafe)} unsafe")
    for m in members:
        print("   ", m)
    if unsafe:
        print("   UNSAFE:", unsafe)



bronze/source=google_qec/google-surface-code-curated.zip: 76 members, 0 unsafe
    README.txt
    surface_code_bX_d3_r25_center_3_5/circuit_detector_error_model.dem
    surface_code_bX_d3_r25_center_3_5/circuit_ideal.stim
    surface_code_bX_d3_r25_center_3_5/circuit_noisy.stim
    surface_code_bX_d3_r25_center_3_5/detection_events.b8
    surface_code_bX_d3_r25_center_3_5/layout.svg
    surface_code_bX_d3_r25_center_3_5/measurements.b8
    surface_code_bX_d3_r25_center_3_5/obs_flips_actual.01
    surface_code_bX_d3_r25_center_3_5/obs_flips_predicted_by_belief_matching.01
    surface_code_bX_d3_r25_center_3_5/obs_flips_predicted_by_correlated_matching.01
    surface_code_bX_d3_r25_center_3_5/obs_flips_predicted_by_pymatching.01
    surface_code_bX_d3_r25_center_3_5/obs_flips_predicted_by_tensor_network_contraction.01
    surface_code_bX_d3_r25_center_3_5/pij_from_even_for_odd.dem
    surface_code_bX_d3_r25_center_3_5/pij_from_odd_for_even.dem
    surface_code_bX_d3_r25_center_3_5/prope

In [6]:
from quantum_lake_student.stages.register_sources import run
result = run("manual-test-1")
print(result)

StageResult(stage='register_sources', run_id='manual-test-1', input_count=3, output_count=3, issue_count=0, started_at=datetime.datetime(2026, 9, 15, 22, 15, 34, 714721, tzinfo=datetime.timezone.utc), finished_at=datetime.datetime(2026, 9, 15, 22, 15, 34, 835738, tzinfo=datetime.timezone.utc))


In [1]:
from pathlib import Path
print("cwd:", Path.cwd())
print("results/part1 exists:", Path("results/part1").exists())
print(sorted(p.name for p in Path("results/part1").glob("*")))


cwd: /workspace/notebooks
results/part1 exists: True
['row_counts.json', 'run.json', 'source_registration.parquet']


## Content-level sampling

So far we've only verified that the *files* are the ones the manifest promises (sizes, hashes, member lists). Before writing any Silver parsing code we need to look *inside* one record from each source and check the actual shape against what `data-sources.md` claims.


In [7]:
#one syndrome CSV row (qec_syndromes)

import ast
import csv
import io

syndrome_object = next(name for name in raw_bytes if "source=qec_syndromes" in name)
with zipfile.ZipFile(io.BytesIO(raw_bytes[syndrome_object])) as zf:
    csv_members = [m for m in zf.namelist() if m.endswith(".csv")]
    print(f"{len(csv_members)} CSV member(s), sampling: {csv_members[0]}")
    with zf.open(csv_members[0]) as fh:
        text = io.TextIOWrapper(fh, encoding="utf-8")
        reader = csv.DictReader(text)
        print("header:", reader.fieldnames)
        first_row = next(reader)

print("\nraw row:", first_row)

parsed_syndromes = ast.literal_eval(first_row["syndromes"])
print("\nparsed 'syndromes' type:", type(parsed_syndromes))
print("outer length (rounds?):", len(parsed_syndromes))
print("inner lengths (checks per round?):", [len(r) for r in parsed_syndromes])
print("values are 0/1 only:", all(v in (0, 1) for r in parsed_syndromes for v in r))
print("labels value:", first_row["labels"], "quantity:", first_row["quantity"])

# sanity check against the filename's nominal sample count, e.g. d-3_pfr-..._nb-10M.csv
print("\nfilename encodes distance/fault-rate/count:", csv_members[0])


7 CSV member(s), sampling: d-3_pfr-0.000010_nb-10M.csv
header: ['labels', 'syndromes', 'quantity']

raw row: {'labels': '0', 'syndromes': '((0, 0, 0, 0), (0, 0, 0, 0), (0, 0, 0, 0), (0, 0, 0, 0))', 'quantity': '9987291'}

parsed 'syndromes' type: <class 'tuple'>
outer length (rounds?): 4
inner lengths (checks per round?): [4, 4, 4, 4]
values are 0/1 only: True
labels value: 0 quantity: 9987291

filename encodes distance/fault-rate/count: d-3_pfr-0.000010_nb-10M.csv


In [8]:
#row count + quantity reconciliation for that same CSV (still just one file, not all seven)

with zipfile.ZipFile(io.BytesIO(raw_bytes[syndrome_object])) as zf:
    with zf.open(csv_members[0]) as fh:
        text = io.TextIOWrapper(fh, encoding="utf-8")
        reader = csv.DictReader(text)
        rows = list(reader)

print("row count:", len(rows))
total_quantity = sum(int(r["quantity"]) for r in rows)
print("sum(quantity):", f"{total_quantity:,}")
print("distinct labels seen:", sorted(set(r["labels"] for r in rows)))

# does the same syndrome string appear under both labels? (data-sources.md asks this explicitly)
from collections import defaultdict
labels_by_syndrome = defaultdict(set)
for r in rows:
    labels_by_syndrome[r["syndromes"]].add(r["labels"])
both_labels = [s for s, labels in labels_by_syndrome.items() if len(labels) > 1]
print("syndromes seen under both labels:", len(both_labels), "out of", len(labels_by_syndrome), "distinct syndromes")
print("=> so 'syndromes' alone is" , "NOT" if both_labels else "", "a unique key")


row count: 68
sum(quantity): 10,000,000
distinct labels seen: ['0', '1']
syndromes seen under both labels: 0 out of 68 distinct syndromes
=> so 'syndromes' alone is  a unique key


In [9]:
#one Google experiment: properties.yml + one measurement/detector/sweep b8 record + one 01 file

import yaml
from quantum_lake_student.formats import b8_record_bytes, iter_b8_records, parse_01_records

google_object = next(name for name in raw_bytes if "source=google_qec" in name)
with zipfile.ZipFile(io.BytesIO(raw_bytes[google_object])) as zf:
    properties_members = sorted(m for m in zf.namelist() if m.endswith("properties.yml"))
    print(f"{len(properties_members)} experiment(s) found, sampling: {properties_members[0]}")

    experiment_dir = properties_members[0].rsplit("/", 1)[0]
    properties = yaml.safe_load(zf.read(properties_members[0]))
    print("\nproperties.yml:", json.dumps(properties, indent=2, default=str)[:1000])

    companion_files = [m for m in zf.namelist() if m.startswith(experiment_dir + "/")]
    print("\ncompanion files in this experiment dir:")
    for m in sorted(companion_files):
        print("   ", m)


5 experiment(s) found, sampling: surface_code_bX_d3_r25_center_3_5/properties.yml

properties.yml: {
  "type": "surface_code_memory_experiment",
  "basis": "X",
  "rounds": 25,
  "distance": 3,
  "data_qubits": 9,
  "measure_qubits": 8,
  "shots": 50000,
  "center_data_qubit_row": 3,
  "center_data_qubit_col": 5,
  "circuit_measurements": 209,
  "circuit_sweep_bits": 9,
  "circuit_detectors": 200,
  "circuit_observables": 1,
  "circuit_qubits": 17
}

companion files in this experiment dir:
    surface_code_bX_d3_r25_center_3_5/circuit_detector_error_model.dem
    surface_code_bX_d3_r25_center_3_5/circuit_ideal.stim
    surface_code_bX_d3_r25_center_3_5/circuit_noisy.stim
    surface_code_bX_d3_r25_center_3_5/detection_events.b8
    surface_code_bX_d3_r25_center_3_5/layout.svg
    surface_code_bX_d3_r25_center_3_5/measurements.b8
    surface_code_bX_d3_r25_center_3_5/obs_flips_actual.01
    surface_code_bX_d3_r25_center_3_5/obs_flips_predicted_by_belief_matching.01
    surface_code_bX_d

In [8]:
import pandas as pd
pd.read_parquet("../results/part1/source_registration.parquet")


,source_name,bronze_object,input_sha256,size_bytes,member_count
0,google_qec,bronze/source=google_qec/google-surface-code-c...,5d6a24f89f055883a49910979490be4baef54d28bf9a0f...,14638673,76
1,qasmbench,bronze/source=qasmbench/qasmbench-qec.zip,60307f88e34b1f752b94223d6d136da72c629b4b625ac6...,144172,19
2,qec_syndromes,bronze/source=qec_syndromes/syndromes_dataset.zip,bdfce36a71f04295ac78fb372d9c2e381801c05e3be119...,358017,8


In [10]:
#decode one shot's worth of bits from each b8 file + confirm the 01 file line count

# fill these in from whatever keys properties.yml actually used once cell above prints them
shots = properties.get("shots")
measurement_count = properties.get("measurement_count") or properties.get("measurements")
detector_count = properties.get("detector_count") or properties.get("detectors")
print("shots:", shots, "measurement_count:", measurement_count, "detector_count:", detector_count)

with zipfile.ZipFile(io.BytesIO(raw_bytes[google_object])) as zf:
    measurements_path = f"{experiment_dir}/measurements.b8"
    detection_events_path = f"{experiment_dir}/detection_events.b8"
    actual_flips_path = f"{experiment_dir}/obs_flips_actual.01"

    measurements_bytes = zf.read(measurements_path)
    detection_bytes = zf.read(detection_events_path)
    actual_flips_bytes = zf.read(actual_flips_path)

expected_record_bytes = b8_record_bytes(measurement_count)
print("\nmeasurements.b8: total bytes =", len(measurements_bytes),
      "| expected record bytes =", expected_record_bytes,
      "| implied shots =", len(measurements_bytes) / expected_record_bytes if expected_record_bytes else None)

first_measurement_record = next(iter_b8_records(measurements_bytes, bits_per_record=measurement_count))
print("first shot's measurement bits (len", len(first_measurement_record), "):", first_measurement_record[:32], "...")

actual_flips = parse_01_records(actual_flips_bytes)
print("\nobs_flips_actual.01: line count =", len(actual_flips), "| matches shots from properties.yml:", len(actual_flips) == shots)


shots: 50000 measurement_count: None detector_count: None


TypeError: '<=' not supported between instances of 'NoneType' and 'int'

In [11]:
#one QASMBench circuit (qec_sm_n5 is called out in data-sources.md as the clearest parity-check example)

qasm_object = next(name for name in raw_bytes if "source=qasmbench" in name)
with zipfile.ZipFile(io.BytesIO(raw_bytes[qasm_object])) as zf:
    qasm_members = [m for m in zf.namelist() if m.endswith(".qasm")]
    print(f"{len(qasm_members)} .qasm member(s):")
    for m in sorted(qasm_members):
        print("   ", m)

    target = next((m for m in qasm_members if "qec_sm_n5" in m), qasm_members[0])
    print("\nsampling:", target)
    text = zf.read(target).decode("utf-8")

print(text)


6 .qasm member(s):
    small/error_correctiond3_n5/error_correctiond3_n5.qasm
    small/error_correctiond3_n5/error_correctiond3_n5_transpiled.qasm
    small/qec_en_n5/qec_en_n5.qasm
    small/qec_en_n5/qec_en_n5_transpiled.qasm
    small/qec_sm_n5/qec_sm_n5.qasm
    small/qec_sm_n5/qec_sm_n5_transpiled.qasm

sampling: small/qec_sm_n5/qec_sm_n5.qasm
// Repetition code syndrome measurement
OPENQASM 2.0;
include "qelib1.inc";
qreg q[3];
qreg a[2];
creg c[3];
creg syn[2];
gate syndrome d1,d2,d3,a1,a2 
{ 
  cx d1,a1; cx d2,a1; 
  cx d2,a2; cx d3,a2; 
}
x q[0]; // error
barrier q;
syndrome q[0],q[1],q[2],a[0],a[1];
measure a -> syn;
if(syn==1) x q[0];
if(syn==2) x q[2];
if(syn==3) x q[1];
measure q -> c;



## Decision-log starter

Capture answers to the `data-sources.md` "questions to investigate" as they get settled above, and any judgment calls, in a plain Markdown decision log (e.g. `results/part1/decision_log.md`), not just as notebook output. Things to record from the sampling above:

- Does the documented schema match the actual CSV header / properties.yml keys?
- Confirmed: `syndromes` alone is not a unique key (same syndrome can appear under both labels) — what is the chosen key for `experiment_id` / row identity?
- Byte-length reconciliation result for `measurements.b8` / `detection_events.b8` / `sweep.b8` against `properties.yml`.
- Chosen `source_record_id` scheme per source, before writing `prepare_data.py`.
